# Hindi ASR Dataset Generation Pipeline (Version 2)

## Objective

Build a scalable Hindi grocery/dairy dialogue generation pipeline using:

- OpenRouter LLM
- HuggingFace Dataset
- Duplicate Removal
- Metadata Generation
- JSON/JSONL Export
- Veena TTS Integration (Future)



**Internship Project:** Hindi ASR Dataset Generation

In [1]:
import os

folders = [

    "notebooks",

    "data",

    "data/raw",

    "data/processed",

    "audio",

    "outputs",

    "configs",

    "scripts",

    "logs"

]

for folder in folders:

    os.makedirs(folder, exist_ok=True)

print("=" * 60)
print("PROJECT STRUCTURE CREATED SUCCESSFULLY")
print("=" * 60)

for folder in folders:

    print("✓", folder)

PROJECT STRUCTURE CREATED SUCCESSFULLY
✓ notebooks
✓ data
✓ data/raw
✓ data/processed
✓ audio
✓ outputs
✓ configs
✓ scripts
✓ logs


In [2]:
!pip install -q openai datasets pandas tqdm

# Step 1: Import Required Libraries

## Objective

Import all required libraries for:

- OpenRouter API
- Dataset Handling
- JSON Processing
- Progress Tracking
- File Management

These libraries will be used throughout the project.

In [3]:
from openai import OpenAI

import json
import os
import random

import pandas as pd

from tqdm import tqdm

from datasets import load_dataset

print("All libraries imported successfully!")

All libraries imported successfully!


# Step 2: Project Configuration

## Objective

Define all project-level configurations in one place.

This includes:

- OpenRouter model
- Output directory
- Dataset file name
- Domain information
- Language information

Keeping all configurations together makes the pipeline reusable and easy to maintain.

In [4]:
config = {

    "model": "google/gemma-4-31b-it:free",

    "output_dir": "outputs",

    "output_file": "outputs/generated_dataset.json",

    "domain": "dairy_order",

    "language": "hindi"

}

print("Project Configuration Loaded Successfully!")

print()

for key, value in config.items():

    print(f"{key} : {value}")

Project Configuration Loaded Successfully!

model : google/gemma-4-31b-it:free
output_dir : outputs
output_file : outputs/generated_dataset.json
domain : dairy_order
language : hindi


# Step 3: Create Project Directories

## Objective

Create a clean project structure for storing:

- Generated datasets
- Logs
- Checkpoints
- Processed data

A well-organized directory structure makes the project scalable,
maintainable and suitable for research and team collaboration.

In [5]:
folders = [

    "outputs",
    "outputs/logs",
    "outputs/checkpoints",

    "data",
    "data/raw",
    "data/processed"

]

for folder in folders:

    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

print()

for folder in folders:

    print("✓", folder)

Project folders created successfully!

✓ outputs
✓ outputs/logs
✓ outputs/checkpoints
✓ data
✓ data/raw
✓ data/processed


# Step 4: Initialize OpenRouter Client

## Objective

Initialize the OpenRouter API client for generating Hindi dairy
conversations.

The client will be used throughout the project for LLM-based
dataset generation.

## Model

google/gemma-4-31b-it:free

In [6]:
OPENROUTER_API_KEY ="PASTE_YOUR_OPENROUTER_API_KEY"

client = OpenAI(

    base_url="https://openrouter.ai/api/v1",

    api_key=OPENROUTER_API_KEY

)

print("OpenRouter Client Initialized Successfully!")

OpenRouter Client Initialized Successfully!


# Step 5: Test OpenRouter Connection

## Objective

Verify that the OpenRouter client is working correctly by generating
a simple Hindi dairy store sentence.

This step confirms that the API connection and selected model are
configured properly before building the dataset generation pipeline.

In [7]:
print(OPENROUTER_API_KEY)

PASTE_YOUR_OPENROUTER_API_KEY


# Step 5: Test OpenRouter Connection

## Objective

Verify that the OpenRouter client is working correctly by generating
a simple Hindi dairy store sentence.

This confirms that the API connection and selected model are configured
properly before building the dataset generation pipeline.

In [8]:
response = client.chat.completions.create(

    model=config["model"],

    messages=[

        {
            "role": "user",
            "content": "Generate one simple Hindi dairy store sentence."
        }

    ],

    max_tokens=50

)

print("Response Received!")

print()

print(response.choices[0].message.content)

Response Received!

यहाँ एक सरल वाक्य है:

**"मुझे एक लीटर दूध चाहिए।"**
*(Mujhe ek litre doodh chahiye.)*

**English translation:** "I want one litre of milk."


# Step 6: Load HuggingFace Dataset

## Objective

Load the Hindi Dairy Dataset from HuggingFace and inspect its structure.

This dataset will serve as the reference dataset for duplicate removal
and quality analysis.

In [9]:
dataset = load_dataset(

    "shujaAK/hindi-dairy-dataset-veena"

)

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/649 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/139M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/850 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcript', 'domain_tag', 'language_mix', 'sentence_length', 'speaker', 'deepgram_transcript', 'deepgram_wer', 'deepgram_cer'],
        num_rows: 850
    })
})


# Step 7: Explore the HuggingFace Dataset

## Objective

Inspect the structure of the Hindi Dairy Dataset by viewing
basic information and a few sample records.

This helps us understand the schema before building the
generation and duplicate removal pipeline.

## Information

- Total Samples
- Available Features
- Sample Records

In [10]:
train_dataset = dataset["train"]

print("Total Samples :", len(train_dataset))

print()

print("Available Features :")

for feature in train_dataset.features:

    print("-", feature)

Total Samples : 850

Available Features :
- audio
- transcript
- domain_tag
- language_mix
- sentence_length
- speaker
- deepgram_transcript
- deepgram_wer
- deepgram_cer


# Step 8: View Sample Records

## Objective

Inspect a few records from the Hindi Dairy Dataset to understand its
structure and metadata.

## We will examine:

- Transcript
- Domain Tag
- Speaker

This information will be used later for conversation generation,
speaker assignment and duplicate removal.

In [11]:
train_dataset = dataset["train"]

for i in range(5):

    print(f"Sample {i+1}")

    print("Transcript :", train_dataset[i]["transcript"])

    print("Domain     :", train_dataset[i]["domain_tag"])

    print("Speaker    :", train_dataset[i]["speaker"])

    print("-" * 50)

Sample 1
Transcript : नमस्ते, क्या यह विजय डेयरी स्टोर है?
Domain     : dairy_order
Speaker    : kavya
--------------------------------------------------
Sample 2
Transcript : हाँ जी, बिलकुल, बताइए मैं आपकी क्या मदद कर सकता हूँ?
Domain     : dairy_order
Speaker    : agastya
--------------------------------------------------
Sample 3
Transcript : मुझे एक लीटर फुल क्रीम दूध चाहिए, अमूल का।
Domain     : dairy_order
Speaker    : maitri
--------------------------------------------------
Sample 4
Transcript : ठीक है, एक लीटर अमूल फुल क्रीम दूध।
Domain     : dairy_order
Speaker    : vinaya
--------------------------------------------------
Sample 5
Transcript : और आधा किलो गाय का दही भी भेज देना।
Domain     : dairy_order
Speaker    : kavya
--------------------------------------------------


# Step 9: Create Existing Transcript Store

## Objective

Extract all existing transcripts from the HuggingFace dataset and store
them inside a Python set.

Using a set provides O(1) lookup time and enables fast duplicate
detection during dataset generation.

## Output

existing_transcripts : set

In [12]:
train_dataset = dataset["train"]

print("Train Dataset Loaded Successfully!")

print()

print("Total Samples :", len(train_dataset))

Train Dataset Loaded Successfully!

Total Samples : 850


# Recovery Cell

Reload the HuggingFace dataset after a runtime reset.

This cell is only used when variables are lost from memory and is not
part of the main pipeline.

In [13]:
from datasets import load_dataset

dataset = load_dataset(
    "shujaAK/hindi-dairy-dataset-veena"
)

train_dataset = dataset["train"]

print("Recovery Successful!")

print("Total Samples :", len(train_dataset))

Recovery Successful!
Total Samples : 850


# Step 9: Create Existing Transcript Store

## Objective

Extract all transcripts from the HuggingFace Hindi Dairy Dataset and
store them inside a Python set.

Using a set enables fast duplicate detection during conversation
generation.

## Output

existing_transcripts : set

## Benefit

Prevents generating sentences that already exist in the reference
dataset.

In [14]:
existing_transcripts = set()

for record in train_dataset:

    existing_transcripts.add(

        record["transcript"].strip()

    )

print("Existing Transcript Store Created Successfully!")

print()

print("Total Existing Transcripts :", len(existing_transcripts))

Existing Transcript Store Created Successfully!

Total Existing Transcripts : 850


# Step 10: View Existing Transcript Samples

## Objective

Inspect a few existing transcripts from the HuggingFace Hindi Dairy
Dataset.

This helps us understand the writing style, vocabulary and dialogue
patterns that already exist in the dataset.

The same information will be useful while generating new conversations
and avoiding duplicates.

In [15]:
print("Total Existing Transcripts :", len(existing_transcripts))

print()

sample_transcripts = list(existing_transcripts)[:10]

for i, transcript in enumerate(sample_transcripts):

    print(f"Sample {i+1}")

    print(transcript)

    print("-" * 60)

Total Existing Transcripts : 850

Sample 1
ठीक है, मैं डिलीवरी वाले भैया से बात करके बताता हूँ, क्या कोई दूसरा समय चलेगा?
------------------------------------------------------------
Sample 2
बिल्कुल, सुनील जी, आपका जीएसटी इनवॉइस तैयार है, आकर ले जाइए।
------------------------------------------------------------
Sample 3
राहुल जी, आपका जो फ्रूट जूस का ऑर्डर था, वो अभी अवेलेबल नहीं है।
------------------------------------------------------------
Sample 4
हाँ, वॉट्सएप पर एड्रेस भेज दीजिए, नाम बता दें उनका?
------------------------------------------------------------
Sample 5
मेरा कुल बिल कितना हुआ?
------------------------------------------------------------
Sample 6
कुल कितना बिल हुआ मेरा?
------------------------------------------------------------
Sample 7
अरे, यह तो कंपनी की गलती है, आप उसका फोटो भेज दीजिए, मैं पैसे वापस कर देता हूँ।
------------------------------------------------------------
Sample 8
अभी-अभी तो पैक हुआ था, कोई बात नहीं, मैं कैंसिल कर देता हूँ।
---------------------

# Step 11: Create Text Cleaning Function

## Objective

Create a reusable function for normalizing Hindi text before duplicate
checking and dataset generation.

The function performs:

- Remove leading spaces
- Remove trailing spaces
- Replace multiple spaces with a single space

## Benefit

Ensures consistent text formatting and improves duplicate detection.

In [16]:
def clean_text(text):

    text = text.strip()

    text = " ".join(text.split())

    return text


print("Text Cleaning Function Created Successfully!")

Text Cleaning Function Created Successfully!


# Step 12: Create Duplicate Checking Function

## Objective

Create a reusable function to identify whether a transcript already
exists in the reference dataset.

The function first cleans the text and then searches inside the
existing transcript store.

## Input

Raw Hindi transcript

## Output

True  → Duplicate

False → Unique

## Benefit

Prevents repeated sentences and improves the overall quality of the
generated ASR dataset.

In [17]:
def is_duplicate(text):

    text = clean_text(text)

    return text in existing_transcripts


print("Duplicate Checking Function Created Successfully!")

Duplicate Checking Function Created Successfully!


# Step 13: Test Duplicate Checking Function

## Objective

Verify that the duplicate checking function correctly identifies
existing and new transcripts.

## Test Cases

1. Existing transcript → Expected Output: True

2. New transcript → Expected Output: False

This validation ensures that duplicate removal works correctly before
integrating it into the generation pipeline.

In [18]:
existing_sentence = "मुझे मीठा दही चाहिए, एक छोटा पैकेट।"

new_sentence = "मुझे दो लीटर ऑर्गेनिक दूध चाहिए।"

print("Existing Sentence :", is_duplicate(existing_sentence))

print("New Sentence      :", is_duplicate(new_sentence))

Existing Sentence : True
New Sentence      : False


# Step 14: Create Transcript Store Update Function

## Objective

Create a reusable function that adds a new unique transcript to the
existing transcript store.

The transcript is first cleaned and then stored in the global set.

## Input

Unique Hindi transcript

## Output

Updated transcript store

## Benefit

As new conversations are generated, they are immediately added to the
reference store, preventing future duplicate generation.

In [19]:
def add_to_transcript_store(text):

    text = clean_text(text)

    existing_transcripts.add(text)

    return len(existing_transcripts)


print("Transcript Store Update Function Created Successfully!")

Transcript Store Update Function Created Successfully!


# Step 15: Test Transcript Store Update Function

## Objective

Verify that a new unique transcript is successfully added to the
existing transcript store.

## Workflow

New Transcript
      │
      ▼
clean_text()
      │
      ▼
add_to_transcript_store()
      │
      ▼
Updated Transcript Count

## Expected Result

The total number of transcripts should increase by one.

In [20]:
before_count = len(existing_transcripts)

new_transcript = "मुझे दो लीटर ऑर्गेनिक दूध चाहिए।"

after_count = add_to_transcript_store(new_transcript)

print("Before Count :", before_count)

print("After Count  :", after_count)

print()

print("Transcript Added Successfully!")

Before Count : 850
After Count  : 851

Transcript Added Successfully!


# Step 16: Create Metadata Generator Function

## Objective

Create a reusable function that converts a transcript into a
structured ASR dataset record.

The function automatically generates:

- transcript
- domain_tag
- language_mix
- sentence_length
- speaker
- source

## Benefit

Every generated sentence will have a consistent format and can be
directly exported to JSON or JSONL for further processing.

In [21]:
def create_record(

    transcript,

    speaker,

    source="openrouter_gemma"

):

    transcript = clean_text(transcript)

    return {

        "transcript": transcript,

        "domain_tag": config["domain"],

        "language_mix": config["language"],

        "sentence_length": len(transcript.split()),

        "speaker": speaker,

        "source": source

    }


print("Metadata Generator Function Created Successfully!")

Metadata Generator Function Created Successfully!


In [22]:
def create_record(

    transcript,

    speaker,

    source="openrouter_gemma"

):

    transcript = clean_text(transcript)

    return {

        "transcript": transcript,

        "domain_tag": config["domain"],

        "language_mix": config["language"],

        "sentence_length": len(transcript.split()),

        "speaker": speaker,

        "source": source

    }


print("create_record() recovered successfully!")

create_record() recovered successfully!


# Step 17: Test Metadata Generator

## Objective

Validate the metadata generator by creating a sample ASR dataset record.

The generated record should contain all required fields in a
consistent format.

## Expected Fields

- transcript
- domain_tag
- language_mix
- sentence_length
- speaker
- source

This confirms that every future generated sentence can be converted
into a structured ASR-ready record.

In [23]:
sample_record = create_record(

    transcript="मुझे एक लीटर ताज़ा दूध चाहिए।",

    speaker="kavya"

)

print("ASR Record Created Successfully!")

print()

for key, value in sample_record.items():

    print(f"{key} : {value}")

ASR Record Created Successfully!

transcript : मुझे एक लीटर ताज़ा दूध चाहिए।
domain_tag : dairy_order
language_mix : hindi
sentence_length : 6
speaker : kavya
source : openrouter_gemma


# Step 18: Initialize Dataset Records

## Objective

Create an empty list to store all generated ASR records.

Each validated and unique record will be appended to this list before
being exported to JSON or JSONL.

## Workflow

Transcript
      │
      ▼
Metadata Generator
      │
      ▼
ASR Record
      │
      ▼
dataset_records[]

In [24]:
dataset_records = []

print("Dataset Record Store Initialized Successfully!")

print()

print("Current Dataset Size :", len(dataset_records))

Dataset Record Store Initialized Successfully!

Current Dataset Size : 0


# Step 19: Append First ASR Record

## Objective

Append a structured ASR record to the dataset record store.

This validates that metadata generation and dataset storage work
together correctly.

## Workflow

Transcript
      │
      ▼
create_record()
      │
      ▼
dataset_records.append()
      │
      ▼
Dataset Size +1

In [25]:
record = create_record(

    transcript="मुझे एक लीटर ताज़ा दूध चाहिए।",

    speaker="kavya"

)

dataset_records.append(record)

print("Record Added Successfully!")

print()

print("Current Dataset Size :", len(dataset_records))

print()

print(dataset_records[0])

Record Added Successfully!

Current Dataset Size : 1

{'transcript': 'मुझे एक लीटर ताज़ा दूध चाहिए।', 'domain_tag': 'dairy_order', 'language_mix': 'hindi', 'sentence_length': 6, 'speaker': 'kavya', 'source': 'openrouter_gemma'}


# Step 20: Create Automatic Record Addition Function

## Objective

Create a reusable function that automatically:

1. Cleans the transcript
2. Checks for duplicates
3. Updates the transcript store
4. Creates metadata
5. Adds the record to the dataset

## Workflow

Raw Transcript
      │
      ▼
Text Cleaning
      │
      ▼
Duplicate Check
      │
      ├── Duplicate → Ignore
      │
      └── Unique
              │
              ▼
Metadata Generation
              │
              ▼
Dataset Append
              │
              ▼
Transcript Store Update

This function will become the core component of the complete
Hindi ASR dataset generation pipeline.

In [26]:
def add_record(

    transcript,

    speaker,

    source="openrouter_gemma"

):

    transcript = clean_text(transcript)

    if is_duplicate(transcript):

        return False

    record = create_record(

        transcript=transcript,

        speaker=speaker,

        source=source

    )

    dataset_records.append(record)

    add_to_transcript_store(transcript)

    return True


print("Automatic Record Addition Function Created Successfully!")

Automatic Record Addition Function Created Successfully!


# Step 21: Test Automatic Record Addition Function

## Objective

Verify that the automatic record addition pipeline correctly handles
both duplicate and unique transcripts.

## Test Cases

Case 1:
Existing transcript
Expected Result → Not Added

Case 2:
New transcript
Expected Result → Successfully Added

## Validation

- Duplicate detection
- Metadata generation
- Dataset append
- Transcript store update

This confirms that the complete ASR data ingestion pipeline works as
expected.

In [27]:
print("Dataset Size Before Test :", len(dataset_records))

print()

# Existing transcript (already added in Step 19)
result1 = add_record(

    "मुझे एक लीटर ताज़ा दूध चाहिए।",

    "kavya"

)

print("Duplicate Record Added :", result1)

# New transcript
result2 = add_record(

    "मुझे आधा किलो ताज़ा पनीर चाहिए।",

    "kavya"

)

print("Unique Record Added :", result2)

print()

print("Dataset Size After Test :", len(dataset_records))

Dataset Size Before Test : 1

Duplicate Record Added : True
Unique Record Added : True

Dataset Size After Test : 3


# Step 22: Synchronize Transcript Store

## Objective

Synchronize the transcript store with all records currently present in
the generated dataset.

This guarantees that duplicate checking always considers both:

1. Existing HuggingFace dataset
2. Newly generated records

## Benefit

Ensures reliable duplicate removal throughout the pipeline.

In [28]:
for record in dataset_records:

    existing_transcripts.add(

        record["transcript"]

    )

print("Transcript Store Synchronized Successfully!")

print()

print("Total Existing Transcripts :", len(existing_transcripts))

Transcript Store Synchronized Successfully!

Total Existing Transcripts : 853


# Step 23: Create Batch Record Addition Function

## Objective

Create a reusable function that processes multiple transcripts at once.

For each transcript:

1. Clean the text
2. Check for duplicates
3. Generate metadata
4. Append to dataset
5. Update transcript store

## Input

List of transcripts

## Output

- Number of records added
- Updated dataset

## Benefit

This function will be used for processing complete conversations
generated by an LLM and will become the main data ingestion module
for the Hindi ASR dataset pipeline.

In [29]:
def add_batch(

    transcripts,

    speaker="kavya",

    source="openrouter_gemma"

):

    added = 0

    for transcript in transcripts:

        success = add_record(

            transcript=transcript,

            speaker=speaker,

            source=source

        )

        if success:

            added += 1

    return added


print("Batch Record Addition Function Created Successfully!")

Batch Record Addition Function Created Successfully!


# Step 24: Test Batch Record Addition Function

## Objective

Validate the batch processing pipeline by adding multiple transcripts
at once.

The function should:

- Ignore duplicate transcripts
- Add only unique transcripts
- Update the dataset automatically

## Test Data

Five dairy-related transcripts containing both existing and new
sentences.

In [30]:
batch = [

    "मुझे एक लीटर ताज़ा दूध चाहिए。",

    "मुझे आधा किलो ताज़ा पनीर चाहिए।",

    "मुझे दो पैकेट मक्खन चाहिए।",

    "दही कितना ताज़ा है?",

    "एक किलो घी पैक कर दीजिए।"

]

before_size = len(dataset_records)

added = add_batch(

    transcripts=batch,

    speaker="kavya"

)

after_size = len(dataset_records)

print("Records Before Batch :", before_size)

print("Records Added        :", added)

print("Records After Batch  :", after_size)

Records Before Batch : 3
Records Added        : 4
Records After Batch  : 7


# Step 25: Generate Dataset Statistics

## Objective

Analyze the generated ASR dataset by computing basic statistics.

The following metrics are calculated:

- Total Records
- Average Sentence Length
- Minimum Sentence Length
- Maximum Sentence Length
- Speaker Distribution
- Domain Distribution

## Benefit

These statistics help verify dataset quality and maintain a balanced
distribution before exporting the final dataset.

In [31]:
from collections import Counter

sentence_lengths = [

    record["sentence_length"]

    for record in dataset_records

]

speaker_counter = Counter(

    record["speaker"]

    for record in dataset_records

)

domain_counter = Counter(

    record["domain_tag"]

    for record in dataset_records

)

print("=" * 50)
print("DATASET STATISTICS")
print("=" * 50)

print("Total Records :", len(dataset_records))

print(
    "Average Length :",
    round(sum(sentence_lengths) / len(sentence_lengths), 2)
)

print("Minimum Length :", min(sentence_lengths))

print("Maximum Length :", max(sentence_lengths))

print()

print("Speaker Distribution")

for speaker, count in speaker_counter.items():

    print(speaker, ":", count)

print()

print("Domain Distribution")

for domain, count in domain_counter.items():

    print(domain, ":", count)

print("=" * 50)

DATASET STATISTICS
Total Records : 7
Average Length : 5.57
Minimum Length : 4
Maximum Length : 6

Speaker Distribution
kavya : 7

Domain Distribution
dairy_order : 7


# Step 26: Export Dataset to JSON

## Objective

Export the generated ASR dataset into a JSON file.

The exported file preserves all metadata and can be used for:

- Dataset versioning
- Future preprocessing
- Veena TTS generation
- Deepgram Teacher ASR
- Whisper fine-tuning

## Output File

outputs/generated_dataset.json

In [32]:
import json
import os

os.makedirs("outputs", exist_ok=True)

output_path = "outputs/generated_dataset.json"

with open(

    output_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        dataset_records,

        f,

        ensure_ascii=False,

        indent=4

    )

print("JSON Export Successful!")

print()

print("Output File :", output_path)

print("Total Records :", len(dataset_records))

JSON Export Successful!

Output File : outputs/generated_dataset.json
Total Records : 7


# Step 27: Export Dataset to JSONL

## Objective

Export the generated ASR dataset in JSON Lines (JSONL) format.

Each line represents one ASR record, making it suitable for:

- Machine Learning pipelines
- HuggingFace datasets
- Streaming data processing
- Whisper fine-tuning
- Large-scale dataset generation

## Output File

outputs/generated_dataset.jsonl

In [33]:
import json

output_path = "outputs/generated_dataset.jsonl"

with open(

    output_path,

    "w",

    encoding="utf-8"

) as f:

    for record in dataset_records:

        f.write(

            json.dumps(

                record,

                ensure_ascii=False

            )

            + "\n"

        )

print("JSONL Export Successful!")

print()

print("Output File :", output_path)

print("Total Records :", len(dataset_records))

JSONL Export Successful!

Output File : outputs/generated_dataset.jsonl
Total Records : 7


# Step 28: Export Dataset to CSV

## Objective

Export the generated ASR dataset into CSV format for easy inspection,
analysis and sharing.

CSV files are useful for:

- Manual verification
- Spreadsheet analysis
- Data annotation
- Quick visualization

## Output File

outputs/generated_dataset.csv

In [34]:
import pandas as pd

df = pd.DataFrame(dataset_records)

output_path = "outputs/generated_dataset.csv"

df.to_csv(

    output_path,

    index=False,

    encoding="utf-8-sig"

)

print("CSV Export Successful!")

print()

print("Output File :", output_path)

print("Total Records :", len(df))

CSV Export Successful!

Output File : outputs/generated_dataset.csv
Total Records : 7


# Step 29: Reload Exported JSON Dataset

## Objective

Reload the exported JSON dataset and verify that all records have been
saved correctly.

This step ensures that the exported file can be reused for future
processing, training and data analysis.

## Input

outputs/generated_dataset.json

## Output

Loaded Python list of ASR records

In [35]:
import json

with open(

    "outputs/generated_dataset.json",

    "r",

    encoding="utf-8"

) as f:

    loaded_dataset = json.load(f)

print("Dataset Reloaded Successfully!")

print()

print("Total Loaded Records :", len(loaded_dataset))

print()

print(loaded_dataset[0])

Dataset Reloaded Successfully!

Total Loaded Records : 7

{'transcript': 'मुझे एक लीटर ताज़ा दूध चाहिए।', 'domain_tag': 'dairy_order', 'language_mix': 'hindi', 'sentence_length': 6, 'speaker': 'kavya', 'source': 'openrouter_gemma'}


# Step 30: Validate Exported Dataset

## Objective

Validate the exported ASR dataset by checking:

- Total records
- Required fields
- Missing values

This ensures that every record follows the expected schema before it
is used for TTS generation or ASR model training.

## Required Fields

- transcript
- domain_tag
- language_mix
- sentence_length
- speaker
- source

In [36]:
required_fields = [

    "transcript",
    "domain_tag",
    "language_mix",
    "sentence_length",
    "speaker",
    "source"

]

valid = True

for record in loaded_dataset:

    for field in required_fields:

        if field not in record:

            valid = False

            print("Missing Field :", field)

print("=" * 50)

print("DATASET VALIDATION")

print("=" * 50)

print("Total Records :", len(loaded_dataset))

print("Validation Status :", "PASSED" if valid else "FAILED")

print("=" * 50)

DATASET VALIDATION
Total Records : 7
Validation Status : PASSED


# Step 31: Create Conversation Parser

## Objective

Convert a raw Hindi dairy shop conversation into individual utterances.

The parser removes:

- Empty lines
- Extra spaces

and returns a clean list of dialogue utterances.

## Input

Raw multi-line conversation

## Output

List of cleaned utterances

This parser will be used before speaker assignment and metadata
generation.

In [37]:
def parse_conversation(conversation):

    utterances = []

    for line in conversation.split("\n"):

        line = clean_text(line)

        if line:

            utterances.append(line)

    return utterances


print("Conversation Parser Created Successfully!")

Conversation Parser Created Successfully!


# Step 32: Create Automatic Speaker Mapper

## Objective

Assign speakers automatically to every utterance in a dairy shop
conversation.

The mapping follows the conversation order:

Utterance 1 → kavya (Customer)

Utterance 2 → agastya (Shopkeeper)

Utterance 3 → kavya

Utterance 4 → agastya

...

## Benefit

This maintains a consistent speaker structure throughout the generated
ASR dataset and matches the HuggingFace Hindi Dairy Dataset format.

In [38]:
def assign_speakers(utterances):

    speaker_records = []

    speakers = ["kavya", "agastya"]

    for index, utterance in enumerate(utterances):

        speaker_records.append({

            "speaker": speakers[index % 2],

            "transcript": utterance

        })

    return speaker_records


print("Automatic Speaker Mapper Created Successfully!")

Automatic Speaker Mapper Created Successfully!


# Step 33: Convert Conversation into ASR Records

## Objective

Build an end-to-end pipeline that converts a raw Hindi dairy shop
conversation into structured ASR dataset records.

## Workflow

Raw Conversation
        │
        ▼
Conversation Parser
        │
        ▼
Utterance List
        │
        ▼
Speaker Assignment
        │
        ▼
Metadata Generation
        │
        ▼
ASR Dataset Records

## Output

List of structured ASR records ready for duplicate checking and
dataset generation.

In [39]:
def conversation_to_records(conversation):

    utterances = parse_conversation(conversation)

    speaker_records = assign_speakers(utterances)

    records = []

    for item in speaker_records:

        record = create_record(

            transcript=item["transcript"],

            speaker=item["speaker"]

        )

        records.append(record)

    return records


print("Conversation to ASR Record Pipeline Created Successfully!")

Conversation to ASR Record Pipeline Created Successfully!


# Step 34: Test Conversation to ASR Record Pipeline

## Objective

Validate the complete conversation processing pipeline by converting a
Hindi dairy shop conversation into structured ASR records.

## Workflow

Raw Conversation
        │
        ▼
Conversation Parser
        │
        ▼
Speaker Assignment
        │
        ▼
Metadata Generation
        │
        ▼
ASR Dataset Records

## Expected Result

Each utterance should become one structured ASR record with all
required metadata.

In [40]:
conversation = """
भैया, एक लीटर दूध देना।
ये लीजिए, और कुछ चाहिए?
आधा किलो दही भी दे दीजिए।
जी, बिल्कुल।
एक पैकेट मक्खन भी दे दीजिए।
ये रहा मक्खन।
कुल कितने रुपये हुए?
दो सौ बीस रुपये।
"""

records = conversation_to_records(conversation)

print("Total ASR Records :", len(records))

print()

for i, record in enumerate(records):

    print(f"Record {i+1}")

    print(record)

    print("-" * 60)

Total ASR Records : 8

Record 1
{'transcript': 'भैया, एक लीटर दूध देना।', 'domain_tag': 'dairy_order', 'language_mix': 'hindi', 'sentence_length': 5, 'speaker': 'kavya', 'source': 'openrouter_gemma'}
------------------------------------------------------------
Record 2
{'transcript': 'ये लीजिए, और कुछ चाहिए?', 'domain_tag': 'dairy_order', 'language_mix': 'hindi', 'sentence_length': 5, 'speaker': 'agastya', 'source': 'openrouter_gemma'}
------------------------------------------------------------
Record 3
{'transcript': 'आधा किलो दही भी दे दीजिए।', 'domain_tag': 'dairy_order', 'language_mix': 'hindi', 'sentence_length': 6, 'speaker': 'kavya', 'source': 'openrouter_gemma'}
------------------------------------------------------------
Record 4
{'transcript': 'जी, बिल्कुल।', 'domain_tag': 'dairy_order', 'language_mix': 'hindi', 'sentence_length': 2, 'speaker': 'agastya', 'source': 'openrouter_gemma'}
------------------------------------------------------------
Record 5
{'transcript': 'एक पै

# Step 35: Production Dataset Generator

## Objective

Create a production-ready function that converts a complete Hindi dairy
conversation into unique ASR dataset records.

## Workflow

Raw Conversation
        │
        ▼
Conversation Parser
        │
        ▼
Speaker Assignment
        │
        ▼
Metadata Generation
        │
        ▼
Duplicate Removal
        │
        ▼
Dataset Update

## Output

Updated dataset_records list containing only unique records.

This function will be used as the main ingestion pipeline for
large-scale Hindi ASR dataset generation.

In [41]:
def process_conversation_dataset(conversation):

    records = conversation_to_records(conversation)

    added = 0

    for record in records:

        if not is_duplicate(record["transcript"]):

            dataset_records.append(record)

            add_to_transcript_store(record["transcript"])

            added += 1

    return added


print("Production Dataset Generator Created Successfully!")

Production Dataset Generator Created Successfully!


# Step 36: Automatic Multi-Conversation Dataset Generator

## Objective

Create a reusable pipeline that processes multiple Hindi dairy
conversations and automatically builds a unique ASR dataset.

## Workflow

Conversation 1
        │
        ▼
Process & Add Unique Records
        │
Conversation 2
        │
        ▼
Process & Add Unique Records
        │
Conversation 3
        │
        ▼
Process & Add Unique Records
        │
        ▼
Final Dataset

## Benefit

This function enables scalable dataset creation and prepares the data
for TTS generation and ASR model training.

In [42]:
def process_multiple_conversations(conversations):

    total_added = 0

    for index, conversation in enumerate(conversations):

        added = process_conversation_dataset(conversation)

        total_added += added

        print(
            f"Conversation {index + 1}: Added {added} records"
        )

    return total_added


print("Automatic Multi-Conversation Generator Created Successfully!")

Automatic Multi-Conversation Generator Created Successfully!


# Step 37: Test Multi-Conversation Generator

## Objective

Validate the automatic multi-conversation pipeline by processing
multiple Hindi dairy shop conversations.

## Workflow

Conversation List
        │
        ▼
process_multiple_conversations()
        │
        ▼
Conversation Parser
        │
        ▼
Speaker Assignment
        │
        ▼
Metadata Generation
        │
        ▼
Duplicate Removal
        │
        ▼
Dataset Update

## Expected Result

Only unique ASR records should be added to the dataset.

In [43]:
conversations = [

"""
भैया, एक लीटर दूध देना।
ये लीजिए।
आधा किलो दही भी दे दीजिए।
जी बिल्कुल।
""",

"""
मुझे आधा किलो पनीर चाहिए।
ताज़ा पनीर अभी आया है।
एक मक्खन का पैकेट भी दे दीजिए।
ये लीजिए।
""",

"""
क्या आपके पास देसी घी है?
हाँ, बिल्कुल है।
एक किलो पैक कर दीजिए।
जी अभी करता हूँ।
"""

]

before_size = len(dataset_records)

added = process_multiple_conversations(conversations)

after_size = len(dataset_records)

print()

print("=" * 50)

print("MULTI-CONVERSATION SUMMARY")

print("=" * 50)

print("Dataset Before :", before_size)

print("Records Added  :", added)

print("Dataset After  :", after_size)

print("=" * 50)

Conversation 1: Added 4 records
Conversation 2: Added 3 records
Conversation 3: Added 4 records

MULTI-CONVERSATION SUMMARY
Dataset Before : 7
Records Added  : 11
Dataset After  : 18


# Step 38: Final Dataset Statistics

## Objective

Analyze the final generated ASR dataset after processing multiple
conversations.

The following statistics are calculated:

- Total Records
- Average Sentence Length
- Minimum Sentence Length
- Maximum Sentence Length
- Speaker Distribution
- Domain Distribution

These statistics help verify dataset quality before TTS generation
and teacher transcription.

In [44]:
from collections import Counter

lengths = [

    record["sentence_length"]

    for record in dataset_records

]

speaker_counter = Counter(

    record["speaker"]

    for record in dataset_records

)

domain_counter = Counter(

    record["domain_tag"]

    for record in dataset_records

)

print("=" * 60)

print("FINAL DATASET STATISTICS")

print("=" * 60)

print("Total Records :", len(dataset_records))

print("Average Length :", round(sum(lengths) / len(lengths), 2))

print("Minimum Length :", min(lengths))

print("Maximum Length :", max(lengths))

print()

print("Speaker Distribution")

for speaker, count in speaker_counter.items():

    print(f"{speaker} : {count}")

print()

print("Domain Distribution")

for domain, count in domain_counter.items():

    print(f"{domain} : {count}")

print("=" * 60)

FINAL DATASET STATISTICS
Total Records : 18
Average Length : 4.94
Minimum Length : 2
Maximum Length : 7

Speaker Distribution
kavya : 13
agastya : 5

Domain Distribution
dairy_order : 18


# Step 41: Create Veena TTS Manifest

## Objective

Create a lightweight manifest from the generated ASR dataset.

The manifest contains only the information required by the TTS
system.

## Fields

- id
- transcript
- speaker

## Benefit

The manifest acts as an intermediate representation between the
dataset generation pipeline and Veena TTS audio synthesis pipeline.

In [45]:
tts_manifest = []

for index, record in enumerate(dataset_records):

    tts_manifest.append({

        "id": f"utt_{index+1:05d}",

        "transcript": record["transcript"],

        "speaker": record["speaker"]

    })

print("Veena TTS Manifest Created Successfully!")

print()

print("Total Manifest Records :", len(tts_manifest))

print()

print(tts_manifest[0])

Veena TTS Manifest Created Successfully!

Total Manifest Records : 18

{'id': 'utt_00001', 'transcript': 'मुझे एक लीटर ताज़ा दूध चाहिए।', 'speaker': 'kavya'}


# Step 42: Export Veena TTS Manifest

## Objective

Export the Veena TTS manifest into JSON format.

This manifest will be used as the input for the Veena TTS engine to
generate synthetic Hindi speech.

## Output

outputs/veena_manifest.json

## Manifest Structure

{
    "id": "utt_00001",
    "transcript": "...",
    "speaker": "kavya"
}

In [46]:
import json
import os

os.makedirs("outputs", exist_ok=True)

manifest_path = "outputs/veena_manifest.json"

with open(

    manifest_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        tts_manifest,

        f,

        ensure_ascii=False,

        indent=4

    )

print("Veena Manifest Export Successful!")

print()

print("Output File :", manifest_path)

print("Total Records :", len(tts_manifest))

Veena Manifest Export Successful!

Output File : outputs/veena_manifest.json
Total Records : 18


# Step 43: Create Veena Audio Manifest

## Objective

Extend the Veena manifest by adding audio file information.

This manifest will be directly compatible with the TTS generation
pipeline.

## Fields

- id
- transcript
- speaker
- audio_path

## Output

outputs/veena_audio_manifest.json

## Benefit

After Veena TTS generates speech, every utterance will already have a
corresponding audio filename, making dataset management simple and
consistent.

In [47]:
veena_audio_manifest = []

for item in tts_manifest:

    veena_audio_manifest.append({

        "id": item["id"],

        "transcript": item["transcript"],

        "speaker": item["speaker"],

        "audio_path": f"audio/{item['id']}.wav"

    })

print("Veena Audio Manifest Created Successfully!")

print()

print("Total Records :", len(veena_audio_manifest))

print()

print(veena_audio_manifest[0])

Veena Audio Manifest Created Successfully!

Total Records : 18

{'id': 'utt_00001', 'transcript': 'मुझे एक लीटर ताज़ा दूध चाहिए।', 'speaker': 'kavya', 'audio_path': 'audio/utt_00001.wav'}


# Step 44: Export Veena Audio Manifest

## Objective

Export the Veena audio manifest into JSON format.

This file serves as the final input to the Veena TTS pipeline and
contains both transcript information and expected audio file paths.

## Output

outputs/veena_audio_manifest.json

## Structure

{
    "id": "...",
    "transcript": "...",
    "speaker": "...",
    "audio_path": "audio/utt_00001.wav"
}

In [48]:
import json

audio_manifest_path = "outputs/veena_audio_manifest.json"

with open(

    audio_manifest_path,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        veena_audio_manifest,

        f,

        ensure_ascii=False,

        indent=4

    )

print("Veena Audio Manifest Export Successful!")

print()

print("Output File :", audio_manifest_path)

print("Total Records :", len(veena_audio_manifest))

Veena Audio Manifest Export Successful!

Output File : outputs/veena_audio_manifest.json
Total Records : 18


# Step 45 : Load Existing HuggingFace Dataset

## Objective

Load the existing Hindi dairy dataset from HuggingFace.

This dataset will act as the base dataset. All newly generated
conversations will be checked against it to avoid duplicates before
being appended.

## Dataset

shujaAK/hindi-dairy-dataset-veena

## Expected Output

- Dataset loaded successfully
- Total records
- Available features

In [49]:
from datasets import load_dataset

print("=" * 60)
print("LOADING EXISTING HUGGINGFACE DATASET")
print("=" * 60)

dataset = load_dataset(
    "shujaAK/hindi-dairy-dataset-veena"
)

train_dataset = dataset["train"]

print()

print("Dataset Loaded Successfully!")

print()

print("Total Records :", len(train_dataset))

print()

print("Available Features:")

for column in train_dataset.column_names:
    print("-", column)

LOADING EXISTING HUGGINGFACE DATASET

Dataset Loaded Successfully!

Total Records : 850

Available Features:
- audio
- transcript
- domain_tag
- language_mix
- sentence_length
- speaker
- deepgram_transcript
- deepgram_wer
- deepgram_cer


# Step 46 : Create Existing Transcript Store

## Objective

Create a transcript store from the existing HuggingFace dataset.

The transcript store will be used for duplicate detection during
automatic dataset generation.

Benefits:

- Fast duplicate lookup
- Prevent repeated conversations
- Maintain dataset quality

In [50]:
print("=" * 60)
print("CREATING EXISTING TRANSCRIPT STORE")
print("=" * 60)

existing_transcripts = set()

for record in train_dataset:

    transcript = record["transcript"].strip().lower()

    existing_transcripts.add(transcript)

print()

print("Transcript Store Created Successfully!")

print()

print("Total Existing Transcripts :", len(existing_transcripts))

print()

print("Sample Transcripts:")

for index, text in enumerate(list(existing_transcripts)[:5], start=1):

    print(f"{index}. {text}")

CREATING EXISTING TRANSCRIPT STORE

Transcript Store Created Successfully!

Total Existing Transcripts : 850

Sample Transcripts:
1. ठीक है, मैं डिलीवरी वाले भैया से बात करके बताता हूँ, क्या कोई दूसरा समय चलेगा?
2. बिल्कुल, सुनील जी, आपका जीएसटी इनवॉइस तैयार है, आकर ले जाइए।
3. राहुल जी, आपका जो फ्रूट जूस का ऑर्डर था, वो अभी अवेलेबल नहीं है।
4. हाँ, वॉट्सएप पर एड्रेस भेज दीजिए, नाम बता दें उनका?
5. मेरा कुल बिल कितना हुआ?


# Step 47 : Production Prompt for Dataset Generation

## Objective

Create a production-quality prompt for generating realistic Hindi
grocery and dairy shop conversations.

Generation Rules

- Natural spoken Hindi
- Customer and Shopkeeper interaction
- Short and long utterances
- Real-world scenarios
- No repetitive wording
- No explanations
- Return valid JSON only

Target

100 conversations
8-12 utterances per conversation

In [51]:
SYSTEM_PROMPT = """
You are an expert Hindi dialogue writer.

Generate realistic conversations that happen in Indian grocery and dairy shops.

Rules:

1. Use natural spoken Hindi.

2. Include customer and shopkeeper only.

3. Conversation length: 8-12 utterances.

4. Mix different situations:

- Milk order
- Paneer order
- Butter
- Curd
- Ghee
- Grocery items
- UPI payment
- Cash payment
- Home delivery
- Discount
- Price inquiry
- Stock unavailable
- Festival rush
- Complaint
- Return / exchange

5. Every utterance should sound different.

6. Avoid repetitive wording.

7. Include fillers like:
हाँ,
अच्छा,
ठीक है,
जी,
चलिये,
कोई बात नहीं

8. Return ONLY valid JSON.

Format:

[
{
"conversation":[
"...",
"..."
]
}
]
"""

print("=" * 60)
print("PRODUCTION PROMPT CREATED")
print("=" * 60)

print()

print(SYSTEM_PROMPT[:600])

print()

print("Prompt Length :", len(SYSTEM_PROMPT))

PRODUCTION PROMPT CREATED


You are an expert Hindi dialogue writer.

Generate realistic conversations that happen in Indian grocery and dairy shops.

Rules:

1. Use natural spoken Hindi.

2. Include customer and shopkeeper only.

3. Conversation length: 8-12 utterances.

4. Mix different situations:

- Milk order
- Paneer order
- Butter
- Curd
- Ghee
- Grocery items
- UPI payment
- Cash payment
- Home delivery
- Discount
- Price inquiry
- Stock unavailable
- Festival rush
- Complaint
- Return / exchange

5. Every utterance should sound different.

6. Avoid repetitive wording.

7. Include fillers like:
हाँ,
अच्छा,
ठीक ह

Prompt Length : 705


# Step 48 : Batch Configuration

## Objective

Configure batch generation for scalable dataset creation.

Instead of generating thousands of utterances in a single API call,
the dataset will be generated in small validated batches.

Benefits

- Better JSON quality
- Easier recovery
- Lower API failure rate
- Easier duplicate removal

In [52]:
# ==========================
# BATCH CONFIGURATION
# ==========================

BATCH_CONVERSATIONS = 10

MIN_UTTERANCES = 8

MAX_UTTERANCES = 12

TARGET_TOTAL_RECORDS = 5000

CURRENT_RECORDS = len(existing_transcripts)

TARGET_NEW_RECORDS = TARGET_TOTAL_RECORDS - CURRENT_RECORDS

print("=" * 60)
print("BATCH CONFIGURATION")
print("=" * 60)

print("Existing Records      :", CURRENT_RECORDS)

print("Target Total Records  :", TARGET_TOTAL_RECORDS)

print("Need To Generate      :", TARGET_NEW_RECORDS)

print("Conversations / Batch :", BATCH_CONVERSATIONS)

print("Utterances / Conv     :", f"{MIN_UTTERANCES}-{MAX_UTTERANCES}")

BATCH CONFIGURATION
Existing Records      : 850
Target Total Records  : 5000
Need To Generate      : 4150
Conversations / Batch : 10
Utterances / Conv     : 8-12


# Step 49 : Conversation Validator

## Objective

Validate every generated conversation before adding it to the dataset.

Checks:

- Conversation is a list
- Length between 8 and 12 utterances
- No empty utterances
- No duplicate utterances
- Natural sentence length

Only validated conversations will be processed further.

In [53]:
# ==========================
# CONVERSATION VALIDATOR
# ==========================

def validate_conversation(conversation):

    if not isinstance(conversation, list):
        return False

    if len(conversation) < MIN_UTTERANCES:
        return False

    if len(conversation) > MAX_UTTERANCES:
        return False

    local_set = set()

    for sentence in conversation:

        sentence = sentence.strip()

        if sentence == "":
            return False

        if sentence.lower() in local_set:
            return False

        local_set.add(sentence.lower())

    return True


print("=" * 60)
print("CONVERSATION VALIDATOR CREATED")
print("=" * 60)

CONVERSATION VALIDATOR CREATED


# Step 50 : Initialize OpenRouter Client

## Objective

Initialize the OpenRouter client that will be used for generating
realistic Hindi grocery and dairy conversations.

The client will later be used for automatic batch generation.

In [58]:
from openai import OpenAI

OPENROUTER_API_KEY = "PASTE_YOUR_OPENROUTER_API_KEY"

client = OpenAI(

    base_url="https://openrouter.ai/api/v1",

    api_key=OPENROUTER_API_KEY

)

print("=" * 60)
print("OPENROUTER CLIENT INITIALIZED SUCCESSFULLY")
print("=" * 60)

OPENROUTER CLIENT INITIALIZED SUCCESSFULLY


# Step 51 : Production User Prompt

## Objective

Create a structured prompt for generating realistic Indian grocery
and dairy store conversations.

The prompt is designed for ASR dataset creation and focuses on
natural spoken Hindi.

Target

5 conversations

8-12 utterances each

JSON output only.

In [55]:
USER_PROMPT = """
Generate 5 unique conversations for Hindi ASR training.

Context:
Indian grocery and dairy shop.

Requirements:

- Customer and shopkeeper only.

- Natural spoken Hindi.

- 8 to 12 utterances.

- Every conversation must be different.

Possible situations:

Milk order
Paneer order
Curd order
Butter
Ghee
Grocery items
UPI payment
Cash payment
Discount
Home delivery
Festival shopping
Stock unavailable
Complaint
Return

Use fillers naturally:
हाँ
अच्छा
ठीक है
जी
चलिये
कोई बात नहीं

Return ONLY this JSON format.

{
  "conversations":[
      {
          "conversation":[
              "...",
              "...",
              "..."
          ]
      }
  ]
}

No markdown.

No explanation.

No numbering.
"""

print("=" * 60)
print("USER PROMPT CREATED SUCCESSFULLY")
print("=" * 60)

print()

print(USER_PROMPT[:700])

USER PROMPT CREATED SUCCESSFULLY


Generate 5 unique conversations for Hindi ASR training.

Context:
Indian grocery and dairy shop.

Requirements:

- Customer and shopkeeper only.

- Natural spoken Hindi.

- 8 to 12 utterances.

- Every conversation must be different.

Possible situations:

Milk order
Paneer order
Curd order
Butter
Ghee
Grocery items
UPI payment
Cash payment
Discount
Home delivery
Festival shopping
Stock unavailable
Complaint
Return

Use fillers naturally:
हाँ
अच्छा
ठीक है
जी
चलिये
कोई बात नहीं

Return ONLY this JSON format.

{
  "conversations":[
      {
          "conversation":[
              "...",
              "...",
              "..."
          ]
      }
  ]
}

No markdown.

No explanation.

No numbe


# Step 52 : Generate First Batch

## Objective

Generate the first batch of realistic Hindi grocery and dairy
conversations using OpenRouter.

This batch will be validated before scaling to 5000 records.

In [59]:
response = client.chat.completions.create(

    model="google/gemma-3-27b-it:free",

    temperature=0.9,

    max_tokens=3500,

    messages=[

        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },

        {
            "role": "user",
            "content": USER_PROMPT
        }

    ]

)

generated_text = response.choices[0].message.content

print("=" * 60)
print("FIRST BATCH GENERATED")
print("=" * 60)

print()

print(generated_text)

NotFoundError: Error code: 404 - {'error': {'message': 'This model is unavailable for free. The paid version is available now - use this slug instead: google/gemma-3-27b-it', 'code': 404}, 'user_id': 'user_3FRzDtYDu1TYDWMKQ6lLYU7ejVS'}

In [61]:
response = client.models.list()

print("Available Models:")

for model in response.data[:30]:
    print(model.id)

Available Models:
google/gemini-3.1-flash-image
google/gemini-3-pro-image
cohere/north-mini-code:free
z-ai/glm-5.2
openrouter/fusion
moonshotai/kimi-k2.7-code
~anthropic/claude-fable-latest
anthropic/claude-fable-5
nex-agi/nex-n2-pro
nvidia/nemotron-3.5-content-safety:free
nvidia/nemotron-3-ultra-550b-a55b:free
nvidia/nemotron-3-ultra-550b-a55b
qwen/qwen3.7-plus
minimax/minimax-m3
stepfun/step-3.7-flash
anthropic/claude-opus-4.8-fast
anthropic/claude-opus-4.8
qwen/qwen3.7-max
x-ai/grok-build-0.1
google/gemini-3.5-flash
anthropic/claude-opus-4.7-fast
perceptron/perceptron-mk1
inclusionai/ring-2.6-1t
google/gemini-3.1-flash-lite
openai/gpt-chat-latest
x-ai/grok-4.3
ibm-granite/granite-4.1-8b
mistralai/mistral-medium-3-5
openrouter/owl-alpha
nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free


# Step 53 : Initialize Production Dataset

## Objective

Create an in-memory dataset that starts with the existing HuggingFace
records and will be continuously expanded with newly generated data.

Every successful batch will be appended and saved automatically.

In [62]:
# ==========================
# PRODUCTION DATASET
# ==========================

production_dataset = []

for record in train_dataset:

    production_dataset.append({

        "transcript": record["transcript"],

        "domain_tag": record["domain_tag"],

        "language_mix": record["language_mix"],

        "sentence_length": record["sentence_length"],

        "speaker": record["speaker"],

        "deepgram_transcript": record["deepgram_transcript"],

        "deepgram_wer": record["deepgram_wer"],

        "deepgram_cer": record["deepgram_cer"]

    })

print("=" * 60)
print("PRODUCTION DATASET INITIALIZED")
print("=" * 60)

print()

print("Current Records :", len(production_dataset))

PRODUCTION DATASET INITIALIZED

Current Records : 850


In [66]:
import json

MODEL_NAME = "google/gemma-4-31b-it:free"

SYSTEM_PROMPT = """
You are an expert Hindi dialogue writer.

Generate realistic conversations for Indian grocery and dairy stores.

Rules:

- Spoken Hindi
- Customer and shopkeeper only
- Natural fillers
- No repeated sentences
- No explanations

Return ONLY valid JSON.
"""

USER_PROMPT = """
Generate exactly 5 conversations.

Each conversation:

- 8 to 12 utterances

Mix these scenarios randomly:

Milk
Paneer
Curd
Butter
Ghee
UPI
Cash
Delivery
Discount
Complaint
Festival
Stock unavailable

Return format:

{
  "conversations":[
    {
      "conversation":[
        "...",
        "...",
        "..."
      ]
    }
  ]
}
"""

response = client.chat.completions.create(

    model=MODEL_NAME,

    messages=[

        {
            "role":"system",
            "content":SYSTEM_PROMPT
        },

        {
            "role":"user",
            "content":USER_PROMPT
        }

    ],

    temperature=0.9,

    max_tokens=3000

)

generated_output = response.choices[0].message.content

print("="*60)
print("RAW MODEL OUTPUT")
print("="*60)

print(generated_output)

RAW MODEL OUTPUT
```json
{
  "conversations": [
    {
      "conversation": [
        "Bhaiya, ek kilo paneer dena, ekdum fresh hona chahiye.",
        "Haan ji, abhi aaya hai. Pack kar doon ya khula chahiye?",
        "Pack wala hi de do, par check kar lena date sahi ho.",
        "Aap befikre rahiye, bilkul fresh hai. Aur kuch?",
        "Ek dahi ka bada dabba bhi dena, Amul wala.",
        "Woh toh khatam ho gaya, Mother Dairy chalega?",
        "Chalo theek hai, wahi de do. Kitne hue?",
        "Paneer aur dahi mila kar 240 rupaye hue.",
        "UPI kar doon? QR code kahan hai?",
        "Woh counter par laga hai, scan kar lijiye.",
        "Ho gaya, check kar lijiye.",
        "Haan, aa gaya. Ye lijiye aapka saaman."
      ]
    },
    {
      "conversation": [
        "Namaste, bhaiya! Ghee ka naya stock aaya kya?",
        "Namaste! Haan, kal hi aaya hai. Kaunsa wala chahiye?",
        "Woh organic wala, jo pichli baar liya tha.",
        "Woh toh abhi stock mein nahi hai, sir.